In [1]:
#pragma cling add_include_path("/usr/local/cuda-11.8/include/")
#pragma cling add_library_path("/usr/local/cuda-11.8/lib64/")

In [2]:
#pragma cling load("/usr/local/cuda-11.8/lib64/libcublas.so")
#pragma cling load("/usr/local/cuda-11.8/lib64/libcublasLt.so")
#pragma cling load("/usr/local/cuda-11.8/lib64/libcudart.so")
#pragma cling load("/usr/local/cuda-11.8/lib64/libcufft.so")
#pragma cling load("/usr/local/cuda-11.8/lib64/libcurand.so")
#pragma cling load("/usr/local/cuda-11.8/lib64/libcusolver.so")
#pragma cling load("/usr/local/cuda-11.8/lib64/libcusparse.so")
#pragma cling load("/usr/local/cuda-11.8/lib64/libnvrtc.so")
#pragma cling load("/usr/local/cuda-11.8/lib64/libnppidei.so")
#pragma cling load("/usr/local/cuda-11.8/lib64/libnppif.so")
#pragma cling load("/usr/local/cuda-11.8/lib64/libnppig.so")
#pragma cling load("/usr/local/cuda-11.8/lib64/libnppc.so")
#pragma cling load("/usr/local/cuda-11.8/lib64/libnppim.so")
#pragma cling load("/usr/local/cuda-11.8/lib64/libnppist.so")
#pragma cling load("/usr/local/cuda-11.8/lib64/libnvjpeg.so")
#pragma cling load("/usr/local/cuda-11.8/lib64/libnvblas.so")
#pragma cling load("/usr/local/cuda-11.8/lib64/libnvToolsExt.so")
#pragma cling load("/usr/local/cuda-11.8/lib64/libOpenCL.so")
#pragma cling load("/usr/local/cuda-11.8/lib64/libcupti.so")

In [3]:
#include <stdio.h>
#include <memory>
#include <iostream>
#include <cuda.h>
#include <cuda_runtime.h>

In [4]:
printf(" CUDA Device Query (Runtime API) version (CUDART static linking)\n\n");

int device_Count = 0;
cudaGetDeviceCount(&device_Count);

if (device_Count == 0)
{
    printf("There are no available device(s) that support CUDA\n");
}
else
{
    printf("Detected %d CUDA Capable device(s)\n", device_Count);
}

 CUDA Device Query (Runtime API) version (CUDART static linking)

Detected 1 CUDA Capable device(s)


## Query Device Information

In [5]:
int device, driver_Version = 0, runtime_Version = 0;

for (device = 0; device < device_Count; ++device)
{
    cudaSetDevice(device);
    cudaDeviceProp device_Property;
    cudaGetDeviceProperties(&device_Property, device);

    printf("\nDevice %d: \"%s\"\n", device, device_Property.name);

    // Console log
    cudaDriverGetVersion(&driver_Version);
    cudaRuntimeGetVersion(&runtime_Version);
    printf("  CUDA Driver Version / Runtime Version          %d.%d / %d.%d\n", driver_Version / 1000, (driver_Version % 100) / 10, runtime_Version / 1000, (runtime_Version % 100) / 10);
    printf("  CUDA Capability Major/Minor version number:    %d.%d\n", device_Property.major, device_Property.minor);
    printf( "  Total amount of global memory:                 %.0f MBytes (%llu bytes)\n",
        (float)device_Property.totalGlobalMem / 1048576.0f, (unsigned long long) device_Property.totalGlobalMem);
    printf("  (%2d) Multiprocessors", device_Property.multiProcessorCount );
    printf("  GPU Max Clock rate:                            %.0f MHz (%0.2f GHz)\n", device_Property.clockRate * 1e-3f, device_Property.clockRate * 1e-6f);

    // This is supported in CUDA 5.0 (runtime API device properties)
    printf("  Memory Clock rate:                             %.0f Mhz\n", device_Property.memoryClockRate * 1e-3f);
    printf("  Memory Bus Width:                              %d-bit\n", device_Property.memoryBusWidth);
    if (device_Property.l2CacheSize)
    {
        printf("  L2 Cache Size:                                 %d bytes\n", device_Property.l2CacheSize);
    }
    printf("  Maximum Texture Dimension Size (x,y,z)         1D=(%d), 2D=(%d, %d), 3D=(%d, %d, %d)\n",
        device_Property.maxTexture1D, device_Property.maxTexture2D[0], device_Property.maxTexture2D[1],
        device_Property.maxTexture3D[0], device_Property.maxTexture3D[1], device_Property.maxTexture3D[2]);
    printf("  Maximum Layered 1D Texture Size, (num) layers  1D=(%d), %d layers\n",
        device_Property.maxTexture1DLayered[0], device_Property.maxTexture1DLayered[1]);
    printf("  Maximum Layered 2D Texture Size, (num) layers  2D=(%d, %d), %d layers\n",
        device_Property.maxTexture2DLayered[0], device_Property.maxTexture2DLayered[1], device_Property.maxTexture2DLayered[2]);
    printf("  Total amount of constant memory:               %lu bytes\n", device_Property.totalConstMem);
    printf("  Total amount of shared memory per block:       %lu bytes\n", device_Property.sharedMemPerBlock);
    printf("  Total number of registers available per block: %d\n", device_Property.regsPerBlock);
    printf("  Warp size:                                     %d\n", device_Property.warpSize);
    printf("  Maximum number of threads per multiprocessor:  %d\n", device_Property.maxThreadsPerMultiProcessor);
    printf("  Maximum number of threads per block:           %d\n", device_Property.maxThreadsPerBlock);
    printf("  Max dimension size of a thread block (x,y,z): (%d, %d, %d)\n",
        device_Property.maxThreadsDim[0],
        device_Property.maxThreadsDim[1],
        device_Property.maxThreadsDim[2]);
    printf("  Max dimension size of a grid size    (x,y,z): (%d, %d, %d)\n",
        device_Property.maxGridSize[0],
        device_Property.maxGridSize[1],
        device_Property.maxGridSize[2]);
    printf("  Maximum memory pitch:                          %lu bytes\n", device_Property.memPitch);
    printf("  Texture alignment:                             %lu bytes\n", device_Property.textureAlignment);
    printf("  Concurrent copy and kernel execution:          %s with %d copy engine(s)\n", (device_Property.deviceOverlap ? "Yes" : "No"), device_Property.asyncEngineCount);
    printf("  Run time limit on kernels:                     %s\n", device_Property.kernelExecTimeoutEnabled ? "Yes" : "No");
    printf("  Integrated GPU sharing Host Memory:            %s\n", device_Property.integrated ? "Yes" : "No");
    printf("  Support host page-locked memory mapping:       %s\n", device_Property.canMapHostMemory ? "Yes" : "No");
    printf("  Alignment requirement for Surfaces:            %s\n", device_Property.surfaceAlignment ? "Yes" : "No");
    printf("  Device has ECC support:                        %s\n", device_Property.ECCEnabled ? "Enabled" : "Disabled");
#if defined(WIN32) || defined(_WIN32) || defined(WIN64) || defined(_WIN64)
    printf("  CUDA Device Driver Mode (TCC or WDDM):         %s\n", device_Property.tccDriver ? "TCC (Tesla Compute Cluster Driver)" : "WDDM (Windows Display Driver Model)");
#endif
    printf("  Device supports Unified Addressing (UVA):      %s\n", device_Property.unifiedAddressing ? "Yes" : "No");
    printf("  Supports Cooperative Kernel Launch:            %s\n", device_Property.cooperativeLaunch ? "Yes" : "No");
    printf("  Supports MultiDevice Co-op Kernel Launch:      %s\n", device_Property.cooperativeMultiDeviceLaunch ? "Yes" : "No");
    printf("  Device PCI Domain ID / Bus ID / location ID:   %d / %d / %d\n", device_Property.pciDomainID, device_Property.pciBusID, device_Property.pciDeviceID);

    const char *sComputeMode[] =
    {
        "Default (multiple host threads can use ::cudaSetDevice() with device simultaneously)",
        "Exclusive (only one host thread in one process is able to use ::cudaSetDevice() with this device)",
        "Prohibited (no host thread can use ::cudaSetDevice() with this device)",
        "Exclusive Process (many threads in one process is able to use ::cudaSetDevice() with this device)",
        "Unknown",
        NULL
    };
    printf("  Compute Mode:\n");
    printf("     < %s >\n", sComputeMode[device_Property.computeMode]);
}	


Device 0: "NVIDIA GeForce RTX 3070 Ti Laptop GPU"
  CUDA Driver Version / Runtime Version          13.2 / 11.8
  CUDA Capability Major/Minor version number:    8.6
  Total amount of global memory:                 8192 MBytes (8589410304 bytes)
  (46) Multiprocessors  GPU Max Clock rate:                            1410 MHz (1.41 GHz)
  Memory Clock rate:                             7001 Mhz
  Memory Bus Width:                              256-bit
  L2 Cache Size:                                 4194304 bytes
  Maximum Texture Dimension Size (x,y,z)         1D=(131072), 2D=(131072, 65536), 3D=(16384, 16384, 16384)
  Maximum Layered 1D Texture Size, (num) layers  1D=(32768), 2048 layers
  Maximum Layered 2D Texture Size, (num) layers  2D=(32768, 32768), 2048 layers
  Total amount of constant memory:               65536 bytes
  Total amount of shared memory per block:       49152 bytes
  Total number of registers available per block: 65536
  Warp size:                                     

In [6]:
%%file gpu_api.h

#pragma once

#ifdef __cplusplus
extern "C" {
#endif

void gpuAdd(int a, int b, int* result);

#ifdef __cplusplus
}
#endif

Overwriting gpu_api.h


In [18]:
%%file gpu_add.cu

#include <iostream>
#include <cuda_runtime.h>
#include "gpu_api.h"

__global__ void gpuAddKernel(int a, int b, int* c) {
    *c = a + b;
}

void gpuAdd(int a, int b, int* result) {
    int* d_c;
    cudaMalloc(&d_c, sizeof(int));

    gpuAddKernel<<<1,1>>>(a, b, d_c);
    cudaMemcpy(result, d_c, sizeof(int), cudaMemcpyDeviceToHost);

    cudaFree(d_c);
    std::cout << "Result from GPU: " << *result << std::endl;
}


Overwriting gpu_add.cu


In [19]:
!nvcc -Xcompiler -fPIC -shared gpu_add.cu -o libgpu.so

In [20]:
#pragma cling load("libgpu.so")

In [21]:
extern "C" void gpuAdd(int a, int b, int* result);

In [28]:
#include <iostream>
#include "gpu_api.h" // or ensure the extern "C" declaration cell was executed
#include <cstdlib>

In [39]:

int result = 0;
gpuAdd(10, 20, &result);
printf("Result from CPU: %d\n", result);


Result from CPU: 30


In [ ]:
!ls -l libgpu.so

-rwxr-xr-x 1 root root 833648 Jun 11 17:45 libgpu.so


In [40]:
!python libgpu.py 9 15

Result from GPU: 24
Result from python: 24
